In [1]:
import tensorflow as tf
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

2024-08-28 02:47:38.829224: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-08-28 02:47:39.269894: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-08-28 02:47:40.296711: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [3]:
actuators = [ "ACCEL_STATE", "STEERING"]
selectedSensors = [ 
        "SPEED",
        "ANGLE_TO_TRACK_AXIS",
        "TRACK_POSITION",
        "TRACK_EDGE_0",
        "TRACK_EDGE_1",
        "OPPONENT_0",
        "OPPONENT_1",
        "OPPONENT_2",
        "OPPONENT_3",
        "OPPONENT_4",
        "OPPONENT_5",
        "OPPONENT_6",
        "OPPONENT_7",
        "OPPONENT_8",
]
dfCollected = pd.read_csv(
        "/home/wbstr/ml_her/ml/logger/aalborg_solo_fast_full.csv",
        sep=',', 
        index_col=False
    )

X = dfCollected[selectedSensors].values
Y_0 = dfCollected["ACCEL_STATE"].values
Y_1 = dfCollected["STEERING"].values

In [4]:
# one hot encoding for the mult class classification node
encode_Y_0 = pd.get_dummies(Y_0)

# Splitting the data into training and testing sets
X_train, X_test, y0_train, y0_test, y1_train, y1_test \
    = train_test_split(X, encode_Y_0, Y_1, test_size=0.2, random_state=42)


In [ ]:
base_nodes = len(selectedSensors)
input_layer = tf.keras.Input(shape=(X_train.shape[1],))
x = tf.keras.layers.Dense(base_nodes * 4, activation='relu')(input_layer)
x = tf.keras.layers.Dense(base_nodes * 2, activation='relu')(x)
x = tf.keras.layers.Dense(base_nodes, activation='relu')(x)

output_layer_pedal = tf.keras.layers.Dense(y0_train.shape[1], activation='softmax', name='pedal_classification')(x)
shape_y1 = 1
output_layer_steering = tf.keras.layers.Dense(shape_y1, activation='tanh', name='steering_regression')(x)
model = tf.keras.Model(inputs=input_layer, outputs=[output_layer_pedal, output_layer_steering])


In [ ]:
# Compile the model
model.compile(optimizer='adam', 
              loss={
                'pedal_classification':tf.keras.losses.CategoricalCrossentropy(), 
                'steering_regression':tf.keras.losses.MeanSquaredError()
                },
              metrics={
                'pedal_classification': [tf.keras.metrics.CategoricalAccuracy()], 
                'steering_regression': [tf.keras.metrics.MeanSquaredError()]})

# Display the model's architecture
model.summary()

In [ ]:
def train_model(model):
    # history = model.fit(X_train, y0_train, epochs=50, batch_size=32, validation_data=(X_test, y0_test))
    history = model.fit(X_train, 
                {'pedal_classification': y0_train, 
                 'steering_regression': y1_train}, 
                epochs=50, 
                batch_size=32, 
                validation_data=(X_test, {'pedal_classification': y0_test, 
                                          'steering_regression': y1_test}))
    
    # Evaluate the model
    results = model.evaluate(X_test, {'pedal_classification': y0_test, 
                                    'steering_regression': y1_test})


    print("fit phase -")
    print(f"Last total loss: {history.history['loss'][-1]}")    
    print(f"Last Pedal Classification Accuracy: {history.history['pedal_classification_categorical_accuracy'][-1]}")
    print(f"Last Steering Regression mse Loss: {history.history['steering_regression_mean_squared_error'][-1]}")

    loss = results[0]
    pedal_classification_acc = results[1]
    steering_regression_loss = results[2]
    print("evaluate phase -")
    print(f"Total Loss: {loss}")
    print(f"Pedal Classification Accuracy: {pedal_classification_acc}")
    print(f"Steering Regression mse Loss: {steering_regression_loss}")


In [6]:
def safe_model(model):
    # save model
    model.save('model_0.keras')
    print("model has been saved!")
def load_model():
    # load model
    model = tf.keras.models.load_model("working_test_model.keras")
    return model

In [ ]:
train_model(model=load_model())

In [ ]:
# safe_model(model=model)

Make predictons 


In [12]:
model = "/home/wbstr/ml_her/ml/model_0.keras"
saved_model = tf.keras.models.load_model(model)

In [15]:
# data is a list of input sensor data 
# example: [136.66106403713096,-0.21474801936180335,0.171515,44.7621,53.9357,200.0,200.0,200.0,200.0,200.0,200.0,200.0,200.0,200.0]
# if it is a single input array put it inside a array -> [[data]]
sensor_data = [ #1,1,0,0,0.5,0.5
    # [124.10731612192016,-0.27645270910841735,-0.093927,42.7057,53.1032,200.0,200.0,29.1245,200.0,106.868,200.0,200.0,200.0,200.0],
    # [124.05056376680213,-0.2932031302490669,-0.0941101,42.159,52.384,200.0,200.0,28.8545,200.0,106.945,200.0,200.0,200.0,200.0],
    # [124.21609885821275,-0.33813161575425044,-0.0942932,41.5575,51.597,200.0,200.0,28.5813,200.0,107.016,200.0,200.0,200.0,200.0],
    # [124.49254299487819,-0.4250687938406308,-0.0944702,40.878,50.712,200.0,200.0,28.3003,200.0,107.079,200.0,200.0,200.0,200.0],
    # [56.658727470035146,3.9796387942637694,-0.126266,20.0786,16.7261,200.0,200.0,200.0,200.0,200.0,200.0,200.0,200.0,200.0], 
    # [56.60124342142637,4.13012615915488,-0.130606,19.9229,16.5889,200.0,200.0,200.0,200.0,200.0,200.0,200.0,200.0,200.0],
    # [89.25983995507309,2.204174367446228,0.293852,58.8644,48.6851,200.0,200.0,200.0,200.0,200.0,200.0,200.0,200.0,200.0],
    # [87.32726143118612,2.2005933812266605,0.290186,58.2996,48.2035,200.0,200.0,200.0,200.0,200.0,200.0,200.0,200.0,200.0]
    [69.15292298152613,0.11480011566359227,0.178241,11.0649,12.2644,200.0,200.0,200.0,200.0,200.0,200.0,200.0,200.0,200.0],
    [133.9220852553408,-0.2220515123763359,0.171985,44.2808,53.1025,200.0,200.0,200.0,200.0,200.0,200.0,200.0,200.0,200.0],
    [139.47204790778403,-0.20820312246802392,0.171045,45.2042,54.7852,200.0,200.0,200.0,200.0,200.0,200.0,200.0,200.0,200.0],
    [131.24910828115992,-0.22878204759573773,0.172437,43.7804,52.2867,200.0,200.0,200.0,200.0,200.0,200.0,200.0,200.0,200.0],
    [128.5141344654846,-0.23594688482384865,0.172839,43.2666,51.4874,200.0,200.0,200.0,200.0,200.0,200.0,200.0,200.0,200.0],
    [125.85716758782769,-0.2440129846637004,0.173242,42.7427,50.7038,200.0,200.0,200.0,200.0,200.0,200.0,200.0,200.0,200.0],
    [123.28320335013494,-0.2516906191184534,0.173645,42.2166,49.9367,200.0,200.0,200.0,200.0,200.0,200.0,200.0,200.0,200.0]
]

p = np.array(sensor_data)
single_p = np.array([list(p[0])])

prediction = saved_model.predict(single_p)
# get value as numeric label -> 0,1,2 -> accelerate, decelerate, brake
pedal_prediction = np.argmax(prediction[0], axis=1)
steer_prediction= prediction[1]
print(f"pedal: {pedal_prediction}\nsteer: {steer_prediction}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
pedal: [1]
steer: [[-0.14525162]]
